In [6]:
"""
====================================================================
ADD THESE CELLS TO YOUR watbalance_prediction.ipynb
แทรกหลัง Cell 80 (หลัง main_workflow definition)
====================================================================
"""

# ====================================================================
# NEW CELL 1: Load N.1 Station Data
# ====================================================================

def load_n1_station_data(filepath='n1_station.csv'):
    """Load and convert N.1 station data to monthly time series"""
    import pandas as pd
    
    print("Loading N.1 station data...")
    df = pd.read_csv(filepath)
    
    # Clean commas and convert to float
    month_cols = ['apr', 'may', 'jun', 'jul', 'aug', 'sep', 
                  'oct', 'nov', 'dec', 'jan', 'feb', 'mar']
    
    for col in month_cols:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    
    # Convert to time series
    records = []
    month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
                 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
    
    for _, row in df.iterrows():
        year = int(row['year'])
        for month_name, month_num in month_map.items():
            date = pd.Timestamp(year=year, month=month_num, day=1)
            records.append({'date': date, 'runoff': row[month_name]})
    
    df_ts = pd.DataFrame(records).set_index('date').sort_index()
    
    print(f"✓ Loaded {len(df_ts)} monthly records ({df_ts.index.min()} to {df_ts.index.max()})")
    print(f"  Runoff range: {df_ts['runoff'].min():.1f} - {df_ts['runoff'].max():.1f} MCM")
    
    return df_ts


# ====================================================================
# NEW CELL 2: Aggregate GEE to Monthly
# ====================================================================

def aggregate_to_monthly(df_daily):
    """Aggregate daily GEE data to monthly"""
    print("Aggregating to monthly...")
    
    agg_dict = {}
    if 'precipitation' in df_daily.columns:
        agg_dict['precipitation'] = 'sum'
    if 'et' in df_daily.columns:
        agg_dict['et'] = 'sum'
    if 'ndvi' in df_daily.columns:
        agg_dict['ndvi'] = 'mean'
    if 'lst' in df_daily.columns:
        agg_dict['lst'] = 'mean'
    if 'soil_moisture' in df_daily.columns:
        agg_dict['soil_moisture'] = 'mean'
    
    df_monthly = df_daily.resample('MS').agg(agg_dict)
    print(f"✓ Aggregated to {len(df_monthly)} months")
    
    return df_monthly


# ====================================================================
# NEW CELL 3: Enhanced Metrics for Hydrology
# ====================================================================

def calculate_hydro_metrics(y_true, y_pred):
    """Calculate NSE, PBIAS, RSR, KGE metrics"""
    from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
    import numpy as np
    
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    # Basic
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE (Nash-Sutcliffe Efficiency)
    nse = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    
    # PBIAS (Percent Bias)
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)
    
    # RSR
    rsr = rmse / np.std(y_true)
    
    # KGE (Kling-Gupta Efficiency)
    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true)
    beta = np.mean(y_pred) / np.mean(y_true)
    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    
    # Volume Error
    ve = 100 * (np.sum(y_pred) - np.sum(y_true)) / np.sum(y_true)
    
    metrics = {
        'RMSE': rmse, 'MAE': mae, 'R²': r2, 'NSE': nse,
        'PBIAS': pbias, 'RSR': rsr, 'KGE': kge, 'Volume_Error': ve
    }
    
    # Print results
    print("\n" + "="*70)
    print("HYDROLOGICAL PERFORMANCE METRICS")
    print("="*70)
    print(f"RMSE:        {rmse:.4f}")
    print(f"MAE:         {mae:.4f}")
    print(f"R²:          {r2:.4f}")
    print(f"NSE:         {nse:.4f}  [{rate_nse(nse)}]")
    print(f"PBIAS:       {pbias:.2f}%  [{rate_pbias(pbias)}]")
    print(f"RSR:         {rsr:.4f}  [{rate_rsr(rsr)}]")
    print(f"KGE:         {kge:.4f}")
    print(f"Volume Err:  {ve:.2f}%")
    print("="*70)
    
    return metrics

def rate_nse(nse):
    if nse > 0.75: return "Very Good"
    elif nse > 0.65: return "Good"
    elif nse > 0.50: return "Satisfactory"
    else: return "Unsatisfactory"

def rate_rsr(rsr):
    if rsr <= 0.50: return "Very Good"
    elif rsr <= 0.60: return "Good"
    elif rsr <= 0.70: return "Satisfactory"
    else: return "Unsatisfactory"

def rate_pbias(pbias):
    abs_pbias = abs(pbias)
    if abs_pbias < 10: return "Very Good"
    elif abs_pbias < 15: return "Good"
    elif abs_pbias < 25: return "Satisfactory"
    else: return "Unsatisfactory"


# ====================================================================
# NEW CELL 4: Modified Main Workflow (MONTHLY VERSION)
# ====================================================================

def main_workflow_monthly():
    """
    Modified workflow using MONTHLY N.1 station data
    แทนที่ main_workflow() เดิม
    """
    
    print("="*60)
    print("RUNOFF PREDICTION - MONTHLY RESOLUTION WITH N.1 DATA")
    print("="*60)
    
    # Initialize extractor
    extractor = GEEDataExtractor(
        Config.BASIN_BOUNDARY,
        Config.START_DATE,
        Config.END_DATE
    )
    
    # ============================================================
    # STEP 1: Extract GEE data (DAILY)
    # ============================================================
    print("\n[STEP 1] Extracting GEE data (this may take a while)...")
    
    data_dict = {
        'precipitation': extractor.extract_precipitation(),
        'et': extractor.extract_evapotranspiration(),
        'ndvi': extractor.extract_ndvi(),
        'lst': extractor.extract_lst(),
        'soil_moisture': extractor.extract_soil_moisture()
    }
    
    # Merge daily GEE data
    preprocessor = DataPreprocessor()
    df_gee_daily = preprocessor.merge_datasets(data_dict)
    df_gee_daily = preprocessor.fill_missing_values(df_gee_daily)
    
    print(f"Daily GEE data: {df_gee_daily.shape}")
    
    # ============================================================
    # STEP 2: Aggregate to MONTHLY
    # ============================================================
    print("\n[STEP 2] Aggregating GEE to monthly...")
    df_gee_monthly = aggregate_to_monthly(df_gee_daily)
    
    # ============================================================
    # STEP 3: Load N.1 station data
    # ============================================================
    print("\n[STEP 3] Loading N.1 station runoff data...")
    df_n1 = load_n1_station_data('n1_station.csv')
    
    # ============================================================
    # STEP 4: Merge datasets
    # ============================================================
    print("\n[STEP 4] Merging N.1 with GEE...")
    df = df_gee_monthly.join(df_n1, how='inner')
    
    print(f"Merged dataset: {df.shape}")
    print(f"Period: {df.index.min()} to {df.index.max()}")
    print(f"Total months: {len(df)}")
    
    # ============================================================
    # STEP 5: Feature Engineering
    # ============================================================
    print("\n[STEP 5] Feature engineering...")
    
    df = preprocessor.add_temporal_features(df)
    df = preprocessor.calculate_water_balance_features(df)
    df = preprocessor.add_lag_features(df, 'runoff', lags=[1, 2, 3, 6, 12])
    
    # Remove NaN from lags
    df = df.dropna()
    
    # Light outlier removal
    df = preprocessor.remove_outliers(df, columns=['runoff'], n_std=4)
    
    print(f"After preprocessing: {df.shape}")
    print(f"\nRunoff statistics (MCM):")
    print(df['runoff'].describe())
    
    # ============================================================
    # STEP 6: Feature Selection
    # ============================================================
    print("\n[STEP 6] Feature selection...")
    
    X_all = df.drop('runoff', axis=1)
    y = df['runoff']
    
    rf_temp = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_temp.fit(X_all, y)
    
    feature_importance = pd.DataFrame({
        'feature': X_all.columns,
        'importance': rf_temp.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Select top 12 features
    top_features = feature_importance.head(12)['feature'].tolist()
    
    print(f"\nTop 12 features:")
    for i, feat in enumerate(top_features, 1):
        imp = feature_importance[feature_importance['feature'] == feat]['importance'].values[0]
        print(f"  {i:2d}. {feat:20s}: {imp:.4f}")
    
    df_reduced = df[top_features + ['runoff']].copy()
    
    # ============================================================
    # STEP 7: Train/Test Split (TEMPORAL)
    # ============================================================
    print("\n[STEP 7] Temporal train/test split...")
    
    # Split by year: Train on 1995-2021, Test on 2022-2024
    df_reduced['year'] = df_reduced.index.year
    
    train_data = df_reduced[df_reduced['year'] <= 2021].drop('year', axis=1)
    test_data = df_reduced[df_reduced['year'] > 2021].drop('year', axis=1)
    
    print(f"Train: {len(train_data)} months (1995-2021)")
    print(f"Test:  {len(test_data)} months (2022-2024)")
    
    # Normalize
    df_scaled, scaler_X, scaler_y = preprocessor.normalize_data(
        df_reduced.drop('year', axis=1), 
        exclude_cols=['runoff'], 
        scale_target=True
    )
    
    # Re-split after scaling
    train_idx = df_reduced[df_reduced['year'] <= 2021].index
    test_idx = df_reduced[df_reduced['year'] > 2021].index
    
    train_data_scaled = df_scaled.loc[train_idx]
    test_data_scaled = df_scaled.loc[test_idx]
    
    X_train = train_data_scaled.drop('runoff', axis=1).values
    y_train = train_data_scaled['runoff'].values
    X_test = test_data_scaled.drop('runoff', axis=1).values
    y_test = test_data_scaled['runoff'].values
    
    # ============================================================
    # STEP 8: Train Model
    # ============================================================
    print("\n[STEP 8] Training model...")
    
    model = SimpleRunoffModel()
    
    # Validation split (last 20% of training data)
    val_size = int(len(X_train) * 0.2)
    X_val = X_train[-val_size:]
    y_val = y_train[-val_size:]
    X_train_final = X_train[:-val_size]
    y_train_final = y_train[:-val_size]
    
    history = model.train(X_train_final, y_train_final, X_val, y_val, 
                         epochs=200, batch_size=16)
    
    # ============================================================
    # STEP 9: Evaluate with Hydrological Metrics
    # ============================================================
    print("\n[STEP 9] Evaluating with hydrological metrics...")
    
    # Get predictions
    y_pred_scaled = model.predict(X_test)
    
    # Inverse transform
    y_true_orig = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_orig = scaler_y.inverse_transform(y_pred_scaled).flatten()
    
    # Calculate metrics
    metrics = calculate_hydro_metrics(y_true_orig, y_pred_orig)
    
    # Plot results
    model.plot_results(y_true_orig, y_pred_orig, dates=test_data_scaled.index)
    
    # ============================================================
    # STEP 10: Seasonal Analysis
    # ============================================================
    print("\n[STEP 10] Seasonal performance analysis...")
    
    # Add month to test data
    test_months = test_data_scaled.index.month
    
    # Wet season: May-Oct (months 5-10)
    wet_mask = test_months.isin([5, 6, 7, 8, 9, 10])
    dry_mask = ~wet_mask
    
    print("\nWET SEASON (May-Oct) Performance:")
    if wet_mask.sum() > 0:
        wet_metrics = calculate_hydro_metrics(
            y_true_orig[wet_mask], 
            y_pred_orig[wet_mask]
        )
    
    print("\nDRY SEASON (Nov-Apr) Performance:")
    if dry_mask.sum() > 0:
        dry_metrics = calculate_hydro_metrics(
            y_true_orig[dry_mask], 
            y_pred_orig[dry_mask]
        )
    
    # ============================================================
    # STEP 11: Save Results
    # ============================================================
    print("\n[STEP 11] Saving results...")
    
    results = {
        'metrics': {k: float(v) for k, v in metrics.items()},
        'top_features': top_features,
        'train_period': f"{train_data.index.min()} to {train_data.index.max()}",
        'test_period': f"{test_data.index.min()} to {test_data.index.max()}",
        'train_size': len(train_data),
        'test_size': len(test_data)
    }
    
    import json
    with open('n1_results.json', 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    print("\n" + "="*60)
    print("WORKFLOW COMPLETED!")
    print("="*60)
    print(f"\nModel Performance (NSE = {metrics['NSE']:.4f})")
    print(f"Files saved: n1_results.json")
    
    return df_reduced, model, metrics, scaler_y


# ====================================================================
# NEW CELL 5: RUN THE WORKFLOW
# ====================================================================

if __name__ == "__main__":
    # Run monthly workflow with N.1 data
    df, model, metrics, scaler_y = main_workflow_monthly()
    
    print("\n✓ Integration with N.1 station data completed!")
    print(f"\nFinal Performance:")
    print(f"  NSE  = {metrics['NSE']:.4f}")
    print(f"  R²   = {metrics['R²']:.4f}")
    print(f"  RMSE = {metrics['RMSE']:.4f} MCM")

RUNOFF PREDICTION - MONTHLY RESOLUTION WITH N.1 DATA


NameError: name 'GEEDataExtractor' is not defined